# Camada Bronze — Ecommerce Clientes

## Objetivo
Realizar a ingestão incremental dos dados brutos da tabela ecommerce_clientes.

## Processamento
- Leitura de arquivos Parquet da camada Raw.
- Polling para identificar novos arquivos.
- Inclusão de metadados de ingestão.
- Tratamento dos tipos de dados.
- Gravação em Delta Lake com modo append.

## Resultado
Disponibilizar os dados na camada Bronze para tratamento e validação na Silver.

In [0]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

print("client_id carregado:", client_id is not None)
print("tenant_id carregado:", tenant_id is not None)
print("client_secret carregado:", client_secret is not None)
print("storage_account carregado:", storage_account is not None)

In [0]:
# Configurações OAuth do Service Principal para acesso ao ADLS

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("✅ Configuração do ADLS preparada com sucesso.")

In [0]:
# ============================================
# CONFIGURAÇÃO DO POLLING - ECOMMERCE_CLIENTES
# ============================================

TABELA = "ecommerce_clientes"
CAMADA = "bronze"

INTERVALO_POLLING = 60  # segundos

print("Tabela:", TABELA)
print("Camada:", CAMADA)
print("Intervalo do polling:", INTERVALO_POLLING, "segundos")

In [0]:
container_squad = "squad1"

In [0]:
def montar_caminho_delta(container_squad, camada, tabela):
    """
    Monta o caminho ADLS para as camadas Bronze, Silver ou Gold.
    """
    
    return (
        f"abfss://{container_squad}@{storage_account}.dfs.core.windows.net/"
        f"{camada}/{tabela}"
    )

In [0]:
bronze_path = montar_caminho_delta(
    container_squad,
    "bronze",
    "ecommerce_clientes"
)

print("Caminho Bronze:")
print(bronze_path)

In [0]:
# ============================================================
# PREPARAR BRONZE CORRIGIDA — SEM GRAVAR
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType
)

# 1. Ler a Bronze atual
df_bronze_atual = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(bronze_path)
)

print("===== BRONZE ATUAL =====")
print("Registros:", df_bronze_atual.count())

# 2. Recuperar o horário de ingestão por arquivo
df_mapa_ingestao = (
    df_bronze_atual
    .groupBy("bronze_source_file")
    .agg(
        F.min("bronze_ingested_at").alias(
            "bronze_ingested_at"
        ),
        F.countDistinct("bronze_ingested_at").alias(
            "_qtd_horarios"
        ),
        F.count("*").alias("_qtd_bronze"),
        F.count("bronze_ingested_at").alias(
            "_qtd_com_horario"
        )
    )
)

# 3. Conferir se os metadados podem ser preservados
problemas = df_mapa_ingestao.filter(
    F.col("bronze_source_file").isNull()
    | (F.trim(F.col("bronze_source_file")) == "")
    | (F.col("_qtd_horarios") != 1)
    | (
        F.col("_qtd_com_horario")
        != F.col("_qtd_bronze")
    )
)

if problemas.limit(1).count() > 0:
    display(problemas)
    raise ValueError(
        "Existem arquivos com metadados ausentes "
        "ou mais de um horário de ingestão."
    )

# 4. Identificar os arquivos originais
arquivos = [
    linha["bronze_source_file"]
    for linha in (
        df_mapa_ingestao
        .select("bronze_source_file")
        .collect()
    )
]

if not arquivos:
    raise ValueError(
        "A Bronze não possui arquivos para reconstruir."
    )

# 5. Definir o schema da Raw
# As datas permanecem em nanossegundos, como Long.
schema_raw = StructType([
    StructField("id_cliente", LongType(), True),
    StructField("uuid_cliente", StringType(), True),
    StructField("nome", StringType(), True),
    StructField("sobrenome", StringType(), True),
    StructField("email", StringType(), True),
    StructField("senha_hash", StringType(), True),
    StructField("dt_cadastro", LongType(), True),
    StructField(
        "dt_ultima_atualizacao",
        LongType(),
        True
    )
])

# 6. Reler somente os arquivos já presentes na Bronze
df_raw_reconstrucao = (
    spark.read
    .options(**adls_options)
    .schema(schema_raw)
    .parquet(*arquivos)
    .withColumn(
        "bronze_source_file",
        F.col("_metadata.file_path")
    )
)

# 7. Comparar as quantidades por arquivo
df_contagem_raw = (
    df_raw_reconstrucao
    .groupBy("bronze_source_file")
    .agg(
        F.count("*").alias("_qtd_raw")
    )
)

df_comparacao = (
    df_mapa_ingestao
    .join(
        df_contagem_raw,
        on="bronze_source_file",
        how="full"
    )
)

divergencias = df_comparacao.filter(
    F.col("_qtd_raw").isNull()
    | F.col("_qtd_bronze").isNull()
    | (
        F.col("_qtd_raw")
        != F.col("_qtd_bronze")
    )
)

if divergencias.limit(1).count() > 0:
    display(divergencias)
    raise ValueError(
        "As quantidades da Raw e da Bronze "
        "diferem por arquivo."
    )

# 8. Preservar o horário original de ingestão
# Criar ano, mês, dia e hora a partir desse horário.
df_bronze_corrigida = (
    df_raw_reconstrucao
    .join(
        df_mapa_ingestao.select(
            "bronze_source_file",
            "bronze_ingested_at"
        ),
        on="bronze_source_file",
        how="inner"
    )
    .withColumn(
        "ano",
        F.year("bronze_ingested_at")
    )
    .withColumn(
        "mes",
        F.month("bronze_ingested_at")
    )
    .withColumn(
        "dia",
        F.dayofmonth("bronze_ingested_at")
    )
    .withColumn(
        "hora",
        F.hour("bronze_ingested_at")
    )
)

# 9. Mostrar os resultados
print("===== SCHEMA CORRIGIDO =====")
df_bronze_corrigida.printSchema()

print("===== CONFERÊNCIA =====")
print("Arquivos conferidos:", len(arquivos))
print(
    "Registros preparados:",
    df_bronze_corrigida.count()
)

display(
    df_bronze_corrigida
    .groupBy("ano", "mes", "dia", "hora")
    .count()
    .orderBy("ano", "mes", "dia", "hora")
)

print("Preparação concluída. Nenhum dado foi gravado.")

In [0]:
# ============================================================
# GRAVAR E VALIDAR A BRONZE CORRIGIDA EM NOVO CAMINHO
# ============================================================

from deltalake import write_deltalake, DeltaTable
from uuid import uuid4
import pyarrow as pa

# 1. Criar um destino exclusivo para esta validação
bronze_validacao_path = (
    bronze_path.rstrip("/")
    + "validacao"
    + uuid4().hex[:12]
)

# 2. Converter os dados para Arrow
dados_arrow = pa.Table.from_pandas(
    df_bronze_corrigida.toPandas(),
    preserve_index=False
)

# Garantir timestamp em microssegundos para o Delta
schema_arrow = pa.schema([
    pa.field(
        campo.name,
        pa.timestamp("us", tz=campo.type.tz)
        if pa.types.is_timestamp(campo.type)
        else campo.type,
        nullable=campo.nullable
    )
    for campo in dados_arrow.schema
])

dados_arrow = dados_arrow.cast(schema_arrow)

# Confirmar que as datas de origem continuam como Long
for coluna in ["dt_cadastro", "dt_ultima_atualizacao"]:
    if dados_arrow.schema.field(coluna).type != pa.int64():
        raise ValueError(f"{coluna} precisa permanecer como Long.")

total_preparado = dados_arrow.num_rows

# 3. Gravar com particionamento físico
write_deltalake(
    bronze_validacao_path,
    dados_arrow,
    mode="error",
    partition_by=["ano", "mes", "dia", "hora"],
    storage_options=delta_storage_options
)

# 4. Ler novamente e validar
tabela_validacao = DeltaTable(
    bronze_validacao_path,
    storage_options=delta_storage_options
)

particoes = tabela_validacao.metadata().partition_columns
dados_gravados = tabela_validacao.to_pyarrow_table()
total_gravado = dados_gravados.num_rows

assert particoes == ["ano", "mes", "dia", "hora"], (
    f"Particionamento inesperado: {particoes}"
)

assert total_gravado == total_preparado, (
    "A quantidade gravada difere da quantidade preparada."
)

for coluna in ["dt_cadastro", "dt_ultima_atualizacao"]:
    assert dados_gravados.schema.field(coluna).type == pa.int64(), (
        f"Tipo incorreto após a gravação: {coluna}"
    )

print("===== VALIDAÇÃO DA NOVA BRONZE =====")
print("Caminho:", bronze_validacao_path)
print("Partições:", particoes)
print("Registros preparados:", total_preparado)
print("Registros gravados:", total_gravado)
print("Datas de origem: Long")
print("VALIDAÇÃO APROVADA!")

In [0]:
silver_path = montar_caminho_delta(
    container_squad,
    "silver",
    "ecommerce_clientes"
)

print("Caminho Silver:")
print(silver_path)

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account}.dfs.core.windows.net",
    credential=credential
)

print("Containers disponíveis:")

for filesystem in service_client.list_file_systems():
    print(filesystem["name"])

In [0]:
# Procurar ecommerce_clientes dentro do container RAW

file_system_client = service_client.get_file_system_client(
    file_system="raw"
)

arquivos_clientes = []

for arquivo in file_system_client.get_paths():
    caminho = arquivo.name

    if "ecommerce_clientes" in caminho.lower():
        arquivos_clientes.append(caminho)

print("===== ARQUIVOS ENCONTRADOS =====")

if arquivos_clientes:
    for caminho in arquivos_clientes:
        print(caminho)
else:
    print("❌ Nenhum arquivo ecommerce_clientes encontrado na RAW")

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType
from pyspark.sql import functions as F

# Schema da tabela ecommerce_clientes
# Datas entram como LongType para contornar TIMESTAMP(NANOS)
schema_clientes = StructType([
    StructField("id_cliente", LongType(), True),
    StructField("uuid_cliente", StringType(), True),
    StructField("nome", StringType(), True),
    StructField("sobrenome", StringType(), True),
    StructField("email", StringType(), True),
    StructField("senha_hash", StringType(), True),
    StructField("dt_cadastro", LongType(), True),
    StructField("dt_ultima_atualizacao", LongType(), True)
])

# Origem dos dados
container_name = "raw"
tabela_alvo = "ecommerce_clientes"

file_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/{tabela_alvo}.parquet"

print("Caminho RAW:")
print(file_path)



In [0]:
# Leitura dos arquivos RAW de ecommerce_clientes

df_clientes_raw = (
    spark.read
    .options(**adls_options)
    .schema(schema_clientes)
    .parquet(file_path)
)

print("✅ RAW lido com sucesso!")
print("Total de registros:", df_clientes_raw.count())

df_clientes_raw.printSchema()
display(df_clientes_raw)

In [0]:
# Converter timestamps de nanossegundos para timestamp

df_clientes = (
    df_clientes_raw
    .withColumn(
        "dt_cadastro",
        F.timestamp_micros(
            (F.col("dt_cadastro") / 1000).cast("long")
        )
    )
    .withColumn(
        "dt_ultima_atualizacao",
        F.timestamp_micros(
            (F.col("dt_ultima_atualizacao") / 1000).cast("long")
        )
    )
)

print("✅ Datas convertidas com sucesso!")
print("Total de registros:", df_clientes.count())

df_clientes.printSchema()
display(df_clientes)

In [0]:
df_bronze = (
    df_clientes

    # Metadados da ingestão
    .withColumn(
        "bronze_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "bronze_ingested_at",
        F.current_timestamp()
    )

    # Particionamento
    .withColumn("ano", F.year("dt_ultima_atualizacao"))
    .withColumn("mes", F.month("dt_ultima_atualizacao"))
    .withColumn("dia", F.dayofmonth("dt_ultima_atualizacao"))
)

print("✅ DataFrame Bronze preparado com metadados e particionamento")


In [0]:
# Filtrar apenas registros inéditos antes de gravar na Bronze

from pyspark.sql import functions as F

try:
    # Ler IDs que já existem na Bronze
    df_ids_bronze = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(bronze_path)
        .select("id_cliente")
        .distinct()
    )

    # Manter somente registros que ainda não existem na Bronze
    df_bronze_novos = (
        df_bronze.alias("raw")
        .join(
            df_ids_bronze.alias("bronze"),
            on="id_cliente",
            how="left_anti"
        )
    )

    print("✅ Filtro de registros inéditos concluído!")
    print("Registros recebidos:", df_bronze.count())
    print("Registros inéditos:", df_bronze_novos.count())

    display(df_bronze_novos)

except Exception as e:
    print("❌ Erro ao verificar registros inéditos:")
    print(e)

In [0]:
# Validação final da camada Bronze

df_bronze_final = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(bronze_path)
)

total_final = df_bronze_final.count()

duplicados_final = (
    df_bronze_final
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("===== VALIDAÇÃO FINAL BRONZE =====")
print("Total de registros:", total_final)
print("IDs duplicados:", duplicados_final)

if total_final > 0 and duplicados_final == 0:
    print("✅ Bronze validada com sucesso!")
else:
    print("⚠️ Verificar os dados da Bronze.")

In [0]:
from pyspark.sql import functions as F

# ============================================================
# RECONSTRUIR RAW COM METADADOS
# Compatível com Databricks Serverless / Unity Catalog
# ============================================================

df_raw_com_metadata = (
    spark.read
    .options(**adls_options)
    .schema(schema_clientes)
    .parquet(file_path)

    # Caminho do arquivo RAW de origem
    .withColumn(
        "bronze_source_file",
        F.col("_metadata.file_path")
    )

    # Momento em que o registro foi ingerido na Bronze
    .withColumn(
        "bronze_ingested_at",
        F.current_timestamp()
    )
)

print("✅ RAW carregada com metadados.")
print("Total encontrado:", df_raw_com_metadata.count())

df_raw_com_metadata.printSchema()

In [0]:
print("===== COMPARAÇÃO ANTES DA MIGRAÇÃO =====")

print("Bronze atual:", df_bronze_final.count())
print("RAW com metadados:", df_raw_com_metadata.count())

print("\nColunas Bronze atual:")
print(df_bronze_final.columns)

print("\nColunas RAW com metadados:")
print(df_raw_com_metadata.columns)

In [0]:
from pyspark.sql import functions as F

# ==========================================
# PREPARAR BRONZE COM METADADOS
# ==========================================

df_bronze_migracao = (
    df_raw_com_metadata

    # Converter timestamps que vieram como nanossegundos
    .withColumn(
        "dt_cadastro",
        F.timestamp_micros(
            (F.col("dt_cadastro") / 1000).cast("long")
        )
    )
    .withColumn(
        "dt_ultima_atualizacao",
        F.timestamp_micros(
            (F.col("dt_ultima_atualizacao") / 1000).cast("long")
        )
    )

    # Particionamento
    .withColumn("ano", F.year("dt_ultima_atualizacao"))
    .withColumn("mes", F.month("dt_ultima_atualizacao"))
    .withColumn("dia", F.dayofmonth("dt_ultima_atualizacao"))
)

print("===== BRONZE PREPARADA PARA MIGRAÇÃO =====")
print("Total:", df_bronze_migracao.count())

df_bronze_migracao.printSchema()

In [0]:
# ==========================================
# CARGA INCREMENTAL - ARQUIVOS NOVOS
# ==========================================

# Arquivos que já foram processados na Bronze
df_arquivos_processados = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(bronze_path)
    .select("bronze_source_file")
    .distinct()
)

# Comparar RAW com Bronze pelo ARQUIVO
df_incremental = (
    df_bronze_migracao.alias("raw")
    .join(
        df_arquivos_processados.alias("bronze"),
        on="bronze_source_file",
        how="left_anti"
    )
)

total_novos = df_incremental.count()

print("===== CARGA INCREMENTAL =====")
print("Novos registros encontrados:", total_novos)

print("Novos arquivos encontrados:")
(
    df_incremental
    .select("bronze_source_file")
    .distinct()
    .show(truncate=False)
)

In [0]:
import time
from datetime import datetime
from pyspark.sql import functions as F

INTERVALO_SEGUNDOS = 60
MAX_EXECUCOES = 3

print("🔄 POLLING INICIADO")
print(f"Intervalo: {INTERVALO_SEGUNDOS} segundos")
print(f"Máximo de execuções: {MAX_EXECUCOES}")


for execucao in range(1, MAX_EXECUCOES + 1):

    print("\n" + "=" * 60)
    print(f"🔎 Polling {execucao}/{MAX_EXECUCOES}")
    print("Horário:", datetime.now())
    print("=" * 60)

    # ========================================================
    # 1. RELER A RAW
    # ========================================================

    df_raw_com_metadata = (
        spark.read
        .options(**adls_options)
        .schema(schema_clientes)
        .parquet(file_path)

        # arquivo de origem
        .withColumn(
            "bronze_source_file",
            F.col("_metadata.file_path")
        )

        # momento da ingestão
        .withColumn(
            "bronze_ingested_at",
            F.current_timestamp()
        )
    )


    # ========================================================
    # 2. VERIFICAR ARQUIVOS JÁ PROCESSADOS
    # ========================================================

    df_arquivos_processados = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(bronze_path)
        .select("bronze_source_file")
        .distinct()
    )


    # ========================================================
    # 3. PEGAR SOMENTE ARQUIVOS NOVOS
    # ========================================================

    df_incremental = (
        df_raw_com_metadata.alias("raw")
        .join(
            df_arquivos_processados.alias("bronze"),
            on="bronze_source_file",
            how="left_anti"
        )
    )

    total_novos = df_incremental.count()

    print(f"📦 Novos registros encontrados: {total_novos}")


    # ========================================================
    # 4. SE EXISTIREM NOVOS DADOS
    # ========================================================

    if total_novos > 0:

        # ----------------------------------------------------
        # CONVERSÃO DAS DATAS
        #
        # RAW:
        # nanosegundos (LONG)
        #
        # BRONZE:
        # TIMESTAMP
        # ----------------------------------------------------

        df_incremental = (
            df_incremental

            .withColumn(
                "dt_cadastro",
                F.timestamp_seconds(
                    (
                        F.col("dt_cadastro")
                        / F.lit(1000000000)
                    ).cast("long")
                )
            )

            .withColumn(
                "dt_ultima_atualizacao",
                F.timestamp_seconds(
                    (
                        F.col("dt_ultima_atualizacao")
                        / F.lit(1000000000)
                    ).cast("long")
                )
            )
        )


        # ====================================================
        # 5. CONFERIR SCHEMA ANTES DA GRAVAÇÃO
        # ====================================================

        print("📋 Schema que será gravado:")

        df_incremental.select(
            "dt_cadastro",
            "dt_ultima_atualizacao"
        ).printSchema()


        # ====================================================
        # 6. GRAVAR NA BRONZE
        # ====================================================

        (
            df_incremental.write
            .format("delta")
            .mode("append")
            .options(**adls_options)
            .save(bronze_path)
        )

        print(
            f"✅ {total_novos} registros adicionados à Bronze!"
        )

    else:

        print("ℹ️ Nenhum dado novo encontrado.")


    # ========================================================
    # 7. AGUARDAR PRÓXIMO POLLING
    # ========================================================

    if execucao < MAX_EXECUCOES:

        print(
            f"⏳ Aguardando {INTERVALO_SEGUNDOS} segundos..."
        )

        time.sleep(INTERVALO_SEGUNDOS)


print("\n🏁 POLLING FINALIZADO")

In [0]:
# ============================================================
# VALIDAÇÃO FINAL - BRONZE ECOMMERCE_CLIENTES
# ============================================================

from pyspark.sql import functions as F

print("=" * 60)
print("🔎 VALIDAÇÃO FINAL DA BRONZE")
print("=" * 60)

# Ler novamente a Bronze atualizada
df_validacao_bronze = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(bronze_path)
)

# 1. Total de registros
total_registros = df_validacao_bronze.count()

# 2. Quantidade de IDs únicos
ids_unicos = (
    df_validacao_bronze
    .select("id_cliente")
    .distinct()
    .count()
)

# 3. Duplicidades
duplicados = total_registros - ids_unicos

# 4. Quantidade de arquivos RAW já processados
arquivos_processados = (
    df_validacao_bronze
    .select("bronze_source_file")
    .distinct()
    .count()
)

print(f"📦 Total de registros na Bronze: {total_registros}")
print(f"👤 IDs únicos: {ids_unicos}")
print(f"👥 Registros duplicados: {duplicados}")
print(f"📁 Arquivos RAW processados: {arquivos_processados}")

# 5. Verificar metadados
print("\n===== METADADOS =====")

df_validacao_bronze.select(
    "id_cliente",
    "bronze_source_file",
    "bronze_ingested_at"
).show(20, truncate=False)

# 6. Verificar registros sem metadados
sem_source_file = (
    df_validacao_bronze
    .filter(F.col("bronze_source_file").isNull())
    .count()
)

sem_ingested_at = (
    df_validacao_bronze
    .filter(F.col("bronze_ingested_at").isNull())
    .count()
)

print("\n===== QUALIDADE DOS METADADOS =====")
print(f"bronze_source_file NULL: {sem_source_file}")
print(f"bronze_ingested_at NULL: {sem_ingested_at}")

print("\n✅ VALIDAÇÃO FINALIZADA")